# **Tracking**
Use this notebook to generate football tracking data from match footage via the trained Pitch Detection Model and Player Detection Model we trained.

---

## Preparation

**Configurations**

In [18]:
GENERATE_VIDEO = 1
TRACK_TEAMS = 1

**Imports**

In [19]:
from ultralytics import YOLO
import supervision as sv
from tqdm import tqdm
import numpy as np
from sports.common.team import TeamClassifier
from sports.configs.soccer import SoccerPitchConfiguration
from sports.common.view import ViewTransformer

In [20]:
from dotenv import load_dotenv
import os

# Locate repo root (directory containing env/, train/, track/, etc.)
_cwd = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_cwd, "env", "keys.env")):
        REPO_ROOT = _cwd
        break
    _cwd = os.path.dirname(_cwd)
else:
    REPO_ROOT = os.path.dirname(os.getcwd())

**Model locations**

Models are loaded from the locally-trained weights under `models/` (produced by `train/train.ipynb`).

In [21]:
# Load locally trained YOLO models
PITCH_MODEL_PATH = os.path.join(REPO_ROOT, "models", "pitch_detection_model_best", "best.pt")
if not os.path.exists(PITCH_MODEL_PATH):
    PITCH_MODEL_PATH = os.path.join(REPO_ROOT, "models", "pitch_detection_model", "weights", "best.pt")

PLAYER_MODEL_PATH = os.path.join(REPO_ROOT, "models", "player_detection_model_best", "best.pt")
if not os.path.exists(PLAYER_MODEL_PATH):
    PLAYER_MODEL_PATH = os.path.join(REPO_ROOT, "models", "player_detection_model", "weights", "best.pt")

print("Pitch model path:", PITCH_MODEL_PATH)
print("Player model path:", PLAYER_MODEL_PATH)

PITCH_DETECTION_MODEL = YOLO(PITCH_MODEL_PATH)
PLAYER_DETECTION_MODEL = YOLO(PLAYER_MODEL_PATH)

Pitch model path: /Users/larry/Bremen/FootballTrackingDataGeneration-main/models/pitch_detection_model_best/best.pt
Player model path: /Users/larry/Bremen/FootballTrackingDataGeneration-main/models/player_detection_model_best/best.pt


---

## Getting Your Models

This notebook now uses the locally trained YOLO weights saved under `models/` (no Roboflow project IDs required).

In [22]:
# PITCH_DETECTION_MODEL is loaded from local weights above.

In [23]:
# PLAYER_DETECTION_MODEL is loaded from local weights above.

---

## Preparing Your Data Source

Update `VIDEO_FILE` with the match footage you want to track

In [24]:
# Name of the video file (including extension) in the local 'footage' folder
VIDEO_FILE = "sample_30.mov"
SOURCE_VIDEO_PATH = os.path.join(REPO_ROOT, "footage", VIDEO_FILE)
print("Using video:", SOURCE_VIDEO_PATH)

Using video: /Users/larry/Bremen/FootballTrackingDataGeneration-main/footage/sample_30.mov


---

## Object Detection

**Classes**

In [25]:
objects = {
    "ball" : 0,
    "goalkeeper" : 1,
    "player" : 2,
    "referee" :3
}

**Annotations**

In [26]:
ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)

**Identify Goal Keeper**

In [27]:
def resolve_goalkeepers_team_id(players, goalkeepers):
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team_1_centroid = players_xy[players.class_id == 1].mean(axis=0)
    goalkeepers_team_id = []
    for goalkeeper_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(goalkeeper_xy - team_0_centroid)
        dist_1 = np.linalg.norm(goalkeeper_xy - team_1_centroid)
        goalkeepers_team_id.append(0 if dist_0 < dist_1 else 1)

    return np.array(goalkeepers_team_id)

**Get Detections**

In [28]:
def get_detections(frame, detections, key_points, tracker, team_classifier):
  CONFIG = SoccerPitchConfiguration()

  # Organize Detections
  ball_detections = detections[detections.class_id == objects["ball"]]
  ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

  all_detections = detections[detections.class_id != objects["ball"]]
  all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
  all_detections = tracker.update_with_detections(detections=all_detections)

  goalkeepers_detections = all_detections[all_detections.class_id == objects["goalkeeper"]]
  players_detections = all_detections[all_detections.class_id == objects["player"]]

  if(team_classifier):
    players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
    players_detections.class_id = team_classifier.predict(players_crops)

  goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
      players_detections, goalkeepers_detections)

  # Adjust Points to 2D Pitch
  filter = key_points.confidence[0] > 0.5
  frame_reference_points = key_points.xy[0][filter]
  pitch_reference_points = np.array(CONFIG.vertices)[filter]

  transformer = ViewTransformer(
      source=frame_reference_points,
      target=pitch_reference_points
  )

  frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
  pitch_ball_xy = transformer.transform_points(points=frame_ball_xy)
  ball_detections.data["pitch_xy"] = pitch_ball_xy

  frame_goalkeepers_xy = goalkeepers_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
  pitch_goalkeepers_xy = transformer.transform_points(points=frame_goalkeepers_xy)
  goalkeepers_detections.data["pitch_xy"] = pitch_goalkeepers_xy

  frame_players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
  pitch_players_xy = transformer.transform_points(points=frame_players_xy)
  players_detections.data["pitch_xy"] = pitch_players_xy

  # Merge Detections
  all_detections = sv.Detections.merge([ players_detections, goalkeepers_detections ])

  return (all_detections, ball_detections)

**Identify Teams**

In [29]:
def generate_team_model(video, PLAYER_DETECTION_MODEL):
  STRIDE = 30

  frame_generator = sv.get_video_frames_generator(
      source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

  crops = []
  for frame in tqdm(frame_generator, desc='collecting crops'):
      results = PLAYER_DETECTION_MODEL(frame, conf=0.3)
      result = results[0]
      detections = sv.Detections.from_ultralytics(result)
      players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
      crops += players_crops

  #team_classifier = TeamClassifier(device="cuda")
  team_classifier = TeamClassifier()
  team_classifier.fit(crops)

  return team_classifier

**Output Results**

In [30]:
def save_tracking_results(players, ball, frames):
  csv = "Frame,Object,Object ID,Team,X1,Y1,X1,X2,X_Pitch,Y_Pitch\n"
  for frame in range(1, frames):
    for player in players:
      player_data = players[player]
      if str(frame) in player_data:
        csv += str(frame) + ",player," + str(player) + "," + str(player_data[str(frame)]["Team"]) + "," + str(player_data[str(frame)]["X1"]) + "," + str(player_data[str(frame)]["Y1"]) + ","  + str(player_data[str(frame)]["X2"]) + "," + str(player_data[str(frame)]["Y2"]) + "," + str(player_data[str(frame)]["X_Pitch"]) + "," + str(player_data[str(frame)]["Y_Pitch"]) + "\n"
    if str(frame) in ball:
      csv += str(frame) + ",ball,,," + str(ball[str(frame)]["X1"]) + "," + str(ball[str(frame)]["Y1"]) + "," + str(ball[str(frame)]["X2"]) + "," + str(ball[str(frame)]["Y2"]) + "," + str(ball[str(frame)]["X_Pitch"]) + "," + str(ball[str(frame)]["Y_Pitch"]) + "\n"

  # Ensure output directory exists
  os.makedirs("./output", exist_ok=True)
  output_path = os.path.join("./output", VIDEO_FILE + ".csv")
  with open(output_path, "w") as file:
    file.write(csv)
  print("Saved tracking CSV to", os.path.abspath(output_path))

**Tracking**

In [31]:
CONFIG = SoccerPitchConfiguration()
tracking_results = ""

#Get Video
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

tracker = sv.ByteTrack()
tracker.reset()

In [32]:
team_classifier = None
if(TRACK_TEAMS):
  team_classifier = generate_team_model(SOURCE_VIDEO_PATH, PLAYER_DETECTION_MODEL)

collecting crops: 0it [00:00, ?it/s]
0: 384x640 1 ball, 25 players, 2 referees, 429.7ms
Speed: 17.4ms preprocess, 429.7ms inference, 11.6ms postprocess per image at shape (1, 3, 384, 640)
collecting crops: 1it [00:02,  2.15s/it]
0: 384x640 20 players, 1 referee, 407.0ms
Speed: 16.4ms preprocess, 407.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
collecting crops: 2it [00:02,  1.18s/it]
0: 384x640 21 players, 3 referees, 367.9ms
Speed: 1.6ms preprocess, 367.9ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
collecting crops: 3it [00:03,  1.19it/s]
0: 384x640 24 players, 3 referees, 705.1ms
Speed: 3.2ms preprocess, 705.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
collecting crops: 4it [00:03,  1.22it/s]
0: 384x640 18 players, 437.6ms
Speed: 2.8ms preprocess, 437.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)
collecting crops: 5it [00:04,  1.40it/s]
0: 384x640 22 players, 2 referees, 402.7ms
Speed: 2.4ms

In [33]:
#Iterate Over Each Frame
frame_number = 1
video_info = sv.VideoInfo.from_video_path(video_path=SOURCE_VIDEO_PATH)
players = {}
ball = {}

with sv.VideoSink(target_path="./output.mp4", video_info=video_info) as sink:
  for frame in tqdm(frame_generator, desc='Collecting Tracking Data...'):
    # Player detection (Ultralytics YOLO detect)
    player_results = PLAYER_DETECTION_MODEL(frame, conf=0.3)
    player_result = player_results[0]
    detections = sv.Detections.from_ultralytics(player_result)

    # Pitch keypoints (Ultralytics YOLO pose)
    pitch_results = PITCH_DETECTION_MODEL(frame, conf=0.3)
    pitch_result = pitch_results[0]
    key_points = sv.KeyPoints.from_ultralytics(pitch_result)

    #Organize Detections
    all_detections, ball_detections = get_detections(frame, detections, key_points, tracker, team_classifier)
    object_ids = all_detections.tracker_id
    team_ids = all_detections.class_id
    object_types = all_detections.data["class_name"]
    pitch_xys = all_detections.data["pitch_xy"]
    ball_pitch_xys = ball_detections.data["pitch_xy"]
    all_detections.class_id = all_detections.class_id.astype(int)

    labels = [
        f"#{tracker_id}"
        for tracker_id
        in all_detections.tracker_id
    ]

    #Iterate Over Frames
    for idx, xyxy in enumerate(all_detections.xyxy):
      team_id = 0
      if(tracker):
        team_id = team_ids[idx]

      object_id = str(object_ids[idx])
      if(object_id not in players):
        players[object_id] = {}

      players[object_id][str(frame_number)] = {
          "Object Type" : object_types[idx],
          "Team" : team_id,
          "X1" : xyxy[0],
          "Y1" : xyxy[1],
          "X2" : xyxy[2],
          "Y2" : xyxy[3],
          "X_Pitch" : pitch_xys[idx][0],
          "Y_Pitch" : pitch_xys[idx][1],
          "Y_MPLSoccer" : float(float(pitch_xys[idx][1]) / float(CONFIG.width)),
          "X_MPLSoccer" : float(float(pitch_xys[idx][0]) / float(CONFIG.length))
      }

    if(ball_detections.xyxy.shape[0]):
      ball[str(frame_number)] = {
            "X1" : ball_detections.xyxy[0][0],
            "Y1" : ball_detections.xyxy[0][1],
            "X2" : ball_detections.xyxy[0][2],
            "Y2" : ball_detections.xyxy[0][3],
            "X_Pitch" : ball_pitch_xys[0][0],
            "Y_Pitch" : ball_pitch_xys[0][1],
            "Y_MPLSoccer" : float(float(ball_pitch_xys[0][1]) / float(CONFIG.width)),
            "X_MPLSoccer" : float(float(ball_pitch_xys[0][0]) / float(CONFIG.length))
      }
    else:
      ball[str(frame_number)] = {
            "X1" : 0,
            "Y1" : 0,
            "X2" : 0,
            "Y2" : 0,
            "X_Pitch" : 0,
            "Y_Pitch" : 0,
            "Y_MPLSoccer" : 0,
            "X_MPLSoccer" : 0
      }

    frame_number += 1

    if(GENERATE_VIDEO):
      annotated_frame = frame.copy()
      annotated_frame = ellipse_annotator.annotate(
          scene=annotated_frame,
          detections=all_detections)
      annotated_frame = label_annotator.annotate(
          scene=annotated_frame,
          detections=all_detections,
          labels=labels)
      annotated_frame = triangle_annotator.annotate(
          scene=annotated_frame,
          detections=ball_detections)

      sink.write_frame(frame=annotated_frame)

0: 384x640 1 ball, 25 players, 2 referees, 552.0ms
Speed: 5.4ms preprocess, 552.0ms inference, 15.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 pitchs, 53.5ms
Speed: 2.2ms preprocess, 53.5ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)
Embedding extraction: 1it [00:01,  1.50s/it]
0: 384x640 1 ball, 26 players, 1 referee, 522.1ms
Speed: 1.5ms preprocess, 522.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 pitchs, 56.6ms
Speed: 2.6ms preprocess, 56.6ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Embedding extraction: 1it [00:01,  1.47s/it]
0: 384x640 25 players, 1 referee, 391.6ms
Speed: 3.8ms preprocess, 391.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 pitchs, 456.4ms
Speed: 4.4ms preprocess, 456.4ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)
Embedding extraction: 1it [00:03,  3.01s/it]
0: 384x640 21 players, 1 referee, 443.1ms
Speed

In [34]:
save_tracking_results(players, ball, frame_number)

Saved tracking CSV to /Users/larry/Bremen/FootballTrackingDataGeneration-main/track/output/sample_30.mov.csv
